# Folkeregister cohort-analyse

Analyse av aldersskohorter fra `folkeregister.cohort_demographics` (Gold-tabell).

## 1. Last data

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('cohort').getOrCreate()

df = spark.read.format('delta').load('s3a://gold/folkeregister/cohort_demographics')
print(f'{df.count():,} rader')
df.printSchema()

## 2. Fordeling per tiår

In [ ]:
import matplotlib.pyplot as plt

pd = df.groupBy('birth_decade').count().orderBy('birth_decade').toPandas()
pd.plot(kind='bar', x='birth_decade', y='count', figsize=(10, 4), legend=False)
plt.title('Antall personer per fødselskull (tiår)')
plt.xlabel('Tiår')
plt.ylabel('Antall personer')
plt.tight_layout()

## 3. Hvor mange er fortsatt i live?

Kombiner med `person_registry` for live-status.

In [ ]:
pr = spark.read.format('delta').load('s3a://silver/folkeregister/person_registry').filter("is_current = true and is_alive = true")
print(f'Levende personer: {pr.count():,}')